<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Prototype merge between cleaned student-course records and student-degree-status snapshots.

**Notebook Shape:** 7 cells (7 code, 0 markdown).

**Inputs / Data Sources:**
- `df_crg=pd.read_parquet('D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_CRG_STUDENT_COURSE\clean_v_crg_student_course.parquet')`
- `df_add=pd.read_parquet(r'D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_ADD_STUDENT_DEGREE_STATUS\clean_v_add_student_degree_status.parquet')`

**Outputs / Side Effects:**
- `No explicit persisted output detected; side effects are limited to notebook display state unless cells are edited.`

**Logic Flow:**
1. Load cleaned CRG and ADD parquet tables.
2. Join on candidate student/degree/term keys.
3. Inspect matched and unmatched records.

**Maintainability Notes:** The notebook has no written output; if this merge is canonical, preserve it in the maintained merge notebook or a script.


In [ ]:
import pandas as pd

JOIN_KEYS = [ "student_id", "degree_id", "part_id"]
df_crg=pd.read_parquet('D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_CRG_STUDENT_COURSE\clean_v_crg_student_course.parquet')
print("="*80)
print("CRG BASIC INFO")
print("="*80)
print("Shape:", df_crg.shape)
print(df_crg.info())
print("\nColumns:")
print(df_crg.columns.tolist())

print("\nCRG join-key null counts:")
print(df_crg[JOIN_KEYS].isna().sum())

print("\nCRG join-key dtypes:")
print(df_crg[JOIN_KEYS].dtypes)

print("\nCRG sample join keys:")
display(df_crg[JOIN_KEYS + ["course_id", "student_course_id"]].head(20))

print("\nCRG unique counts:")
print(df_crg[JOIN_KEYS].nunique(dropna=False))

print("\nCRG duplicate count on student-course grain:")
crg_grain = JOIN_KEYS + ["course_id"]
print(df_crg.duplicated(crg_grain, keep=False).sum())

print("\nCRG duplicate examples on student-course grain:")
display(
    df_crg.loc[df_crg.duplicated(crg_grain, keep=False), 
               crg_grain + ["student_course_id", "grade_id", "final_mark", "points", "finish_status", "register_status"]]
    .sort_values(crg_grain)
    .head(50)
)

In [ ]:
df_add=pd.read_parquet(r'D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_ADD_STUDENT_DEGREE_STATUS\clean_v_add_student_degree_status.parquet')
print("="*80)
print("ADD BASIC INFO")
print("="*80)
print("Shape:", df_add.shape)
print(df_add.info())
print("\nColumns:")
print(df_add.columns.tolist())

print("\nADD join-key null counts:")
print(df_add[JOIN_KEYS].isna().sum())

print("\nADD join-key dtypes:")
print(df_add[JOIN_KEYS].dtypes)

print("\nADD sample join keys:")
display(df_add[JOIN_KEYS + ["student_status_id"]].head(20))

print("\nADD unique counts:")
print(df_add[JOIN_KEYS].nunique(dropna=False))

print("\nADD duplicate count on semester snapshot grain:")
print(df_add.duplicated(JOIN_KEYS, keep=False).sum())

print("\nADD duplicate examples on semester snapshot grain:")
display(
    df_add.loc[df_add.duplicated(JOIN_KEYS, keep=False),
               JOIN_KEYS + ["student_status_id", "finish_status", "start_part_id", "finish_part_id"]]
    .sort_values(JOIN_KEYS)
    .head(50)
)

In [ ]:
id_cols_crg = [ "student_id", "degree_id", "part_id", "course_id", "student_course_id"]
id_cols_add = [ "student_id", "degree_id", "part_id", "student_status_id"]

print("="*80)
print("CRG ID SAMPLE VALUES")
print("="*80)
for col in id_cols_crg:
    if col in df_crg.columns:
        print(f"\n{col}")
        print(df_crg[col].dropna().astype(str).head(20).tolist())

print("="*80)
print("ADD ID SAMPLE VALUES")
print("="*80)
for col in id_cols_add:
    if col in df_add.columns:
        print(f"\n{col}")
        print(df_add[col].dropna().astype(str).head(20).tolist())

In [ ]:
JOIN_KEYS = ["student_id", "degree_id", "part_id"]

print("="*80)
print("ADD UNIQUENESS CHECK")
print("="*80)

print("ADD rows:", len(df_add))
print("ADD unique JOIN_KEYS:", df_add[JOIN_KEYS].drop_duplicates().shape[0])

add_dup_mask = df_add.duplicated(JOIN_KEYS, keep=False)

print("ADD duplicated rows on JOIN_KEYS:", add_dup_mask.sum())
print("ADD duplicated groups on JOIN_KEYS:", df_add.loc[add_dup_mask, JOIN_KEYS].drop_duplicates().shape[0])

display(
    df_add.loc[
        add_dup_mask,
        JOIN_KEYS + [
            "student_status_id",
            "grade_version_id",
            "finish_status",
            "start_part_id",
            "finish_part_id",
            "start_agpa_points",
            "start_agpa_percent",
            "gpa_points",
            "gpa_percent",
            "end_agpa_points",
            "end_agpa_percent",
            "reg_total_semesters",
            "start_level_id",
            "start_level_name_pl",
        ]
    ]
    .sort_values(JOIN_KEYS)
    .head(100)
)

In [ ]:
CRG_GRAIN = ["student_id", "degree_id", "part_id", "course_id"]

print("="*80)
print("CRG COURSE GRAIN CHECK")
print("="*80)

print("CRG rows:", len(df_crg))
print("CRG unique student_course_id:", df_crg["student_course_id"].nunique(dropna=False))
print("CRG duplicated student_course_id rows:", df_crg.duplicated(["student_course_id"], keep=False).sum())

print("CRG unique CRG_GRAIN:", df_crg[CRG_GRAIN].drop_duplicates().shape[0])

crg_dup_mask = df_crg.duplicated(CRG_GRAIN, keep=False)

print("CRG duplicated rows on student-degree-part-course:", crg_dup_mask.sum())
print("CRG duplicated groups:", df_crg.loc[crg_dup_mask, CRG_GRAIN].drop_duplicates().shape[0])

display(
    df_crg.loc[
        crg_dup_mask,
        CRG_GRAIN + [
            "student_course_id",
            "grade_id",
            "final_mark",
            "points",
            "finish_status",
            "course_outcome_status",
            "register_status",
            "attempt_number",
            "attempt_count",
        ]
    ]
    .sort_values(CRG_GRAIN)
    .head(100)
)

In [ ]:
JOIN_KEYS = ["student_id", "degree_id", "part_id"]

add_snapshot_cols = [
    "student_status_id",
    "student_id",
    "degree_id",
    "part_id",

    # safe / start-of-semester features
    "start_agpa_points",
    "start_agpa_percent",
    "reg_total_semesters",
    "start_level_id",
    "start_level_name_pl",
    "start_part_id",

    # keep for audit only, not model input yet
    "finish_part_id",
    "finish_status",
]

df_add_snapshot = df_add[add_snapshot_cols].copy()

print("="*80)
print("PRE-MERGE VALIDATION")
print("="*80)

print("CRG rows before merge:", len(df_crg))
print("ADD snapshot rows:", len(df_add_snapshot))
print("ADD unique JOIN_KEYS:", df_add_snapshot[JOIN_KEYS].drop_duplicates().shape[0])

assert df_add_snapshot.duplicated(JOIN_KEYS).sum() == 0, "ADD is not unique on JOIN_KEYS."

df_merge_test = df_crg.merge(
    df_add_snapshot,
    on=JOIN_KEYS,
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("_crg", "_add")
)

print("\nRows after merge:", len(df_merge_test))
print("Row count changed:", len(df_merge_test) - len(df_crg))

assert len(df_merge_test) == len(df_crg), "Row count changed after merge. Investigate duplicate ADD keys."

print("\nMerge result:")
print(df_merge_test["_merge"].value_counts(dropna=False))

missing_snapshot_mask = df_merge_test["_merge"].eq("left_only")

print("\nMissing ADD snapshot rows:", missing_snapshot_mask.sum())
print("Missing ADD snapshot ratio:", missing_snapshot_mask.mean())

unmatched_snapshot_report = df_merge_test.loc[missing_snapshot_mask].copy()

print("\nUnmatched rows by part_id:")
display(unmatched_snapshot_report["part_id"].value_counts(dropna=False).head(30))

print("\nUnmatched rows by degree_id:")
display(unmatched_snapshot_report["degree_id"].value_counts(dropna=False).head(30))

print("\nUnmatched rows by register_status:")
display(unmatched_snapshot_report["register_status"].value_counts(dropna=False))

print("\nUnmatched rows by CRG finish_status:")
display(unmatched_snapshot_report["finish_status_crg"].value_counts(dropna=False))

print("\nUnmatched sample:")
display(
    unmatched_snapshot_report[
        [
            "student_course_id",
            "student_id",
            "degree_id",
            "part_id",
            "course_id",
            "grade_id",
            "final_mark",
            "points",
            "finish_status_crg",
            "register_status",
        ]
    ].head(100)
)